In [1]:

!pip install numpy
!pip install opencv-python
!pip install ultralytics 
!pip install pandas



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


YOLO POSE → BLENDER PIPELINE

In [2]:
import cv2
import numpy as np
import pandas as pd
import json
from ultralytics import YOLO

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060 Laptop GPU


CONFIGURAÇÃO

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [7]:
VIDEO_PATH = "video.mp4"
#MODEL_PATH = "../Models/yolov8x-pose-p6.pt"

MODEL_PATH = "../Models/yolov8s-pose.pt"

model = YOLO(MODEL_PATH)

# força GPU
model.to(device)

# opcional: melhora performance
torch.backends.cudnn.benchmark = True

cap = cv2.VideoCapture(VIDEO_PATH)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

COLETA DE DADOS (YOLO)

In [8]:
data = []
frame_id = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    #results = model(frame)
    results = model.predict(
        frame,
        device=device,
        verbose=False
    )


    for r in results:
        if r.keypoints is None:
            continue

        # Pega apenas 1 pessoa (primeira detecção)
        kpts = r.keypoints.xy.cpu().numpy()[0]

        for joint_id, (x, y) in enumerate(kpts):
            data.append({
                "frame": frame_id,
                "joint": joint_id,
                "x": float(x),
                "y": float(y)
            })

    frame_id += 1

cap.release()

df = pd.DataFrame(data)

SMOOTHING (FILTRO EXPONENCIAL)

In [9]:
def smooth_series(series, alpha=0.7):
    smoothed = []
    prev = None

    for p in series:
        if prev is None:
            prev = p
        else:
            p = alpha * prev + (1 - alpha) * p
            prev = p
        smoothed.append(p)

    return smoothed

df["x_smooth"] = df.groupby("joint")["x"].transform(lambda s: smooth_series(s))
df["y_smooth"] = df.groupby("joint")["y"].transform(lambda s: smooth_series(s))


NORMALIZAÇÃO (IMPORTANTE PARA BLENDER)

In [10]:
df["x_smooth"] = (df["x_smooth"] - frame_width / 2) / frame_width
df["y_smooth"] = (df["y_smooth"] - frame_height / 2) / frame_height

# inverter Y (OpenCV vs Blender)
df["y_smooth"] *= -1

EXPORTAÇÃO PARA JSON (FORMATO BLENDER)


In [11]:
out = {}

for frame, group in df.groupby("frame"):
    joints = {}

    for _, row in group.iterrows():
        joint_id = int(row["joint"])

        # Z fake simples (opcional)
        z = 0.0

        joints[joint_id] = [
            float(row["x_smooth"]),
            float(row["y_smooth"]),
            z
        ]

    out[int(frame)] = joints

with open("pose.json", "w") as f:
    json.dump(out, f, indent=2)

print("Export finalizado: pose.json")

Export finalizado: pose.json


VIDEO OUTPUT

In [14]:
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter(
    "preview_pose.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (frame_width, frame_height)
)